# Импорты и загрузка данных

In [1]:
import pandas as pd

import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import optuna

import joblib

import os

import json



d:\programming\VSCode\ML\competition\odsAI\kiberPolka\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path_train = "data\\res_data\\train.parquet"
path_test = "data\\res_data\\test.parquet"
path_train_target = "data\\train\\train_target.parquet"

path_models_old = "data\\model\\model_extra_feature2"
path_models_new = "data\\model\\after_optuna"

In [3]:
train = pd.read_parquet(path_train)
train_t = pd.read_parquet(path_train_target)

test = pd.read_parquet(path_test)

In [4]:
target_cols = [col for col in train_t.columns if col != "customer_id"]

# Основные функции

In [5]:
targets_sorted = [
    "target_9_3",   # 0.687142
    "target_9_6",   # 0.690471
    "target_3_1",   # 0.698810
    "target_5_2",   # 0.717398
    "target_6_1",   # 0.724488
    "target_6_2",   # 0.733117
    "target_2_5",   # 0.743992
    "target_5_1",   # 0.754111
    "target_10_1",  # 0.755271
    "target_2_4",   # 0.758797
    "target_2_6",   # 0.761702
    "target_3_3",   # 0.766708
    "target_9_7",   # 0.769281
    "target_6_3",   # 0.775268
    "target_9_1",   # 0.779262
    "target_2_3",   # 0.796625
    "target_7_3",   # 0.804336
    "target_7_1",   # 0.809892
    "target_1_2",   # 0.827695
    "target_2_1",   # 0.829256
    "target_1_4",   # 0.837311
    "target_9_2",   # 0.839142
    "target_6_4",   # 0.841757
    "target_9_5",   # 0.847283
    "target_4_1",   # 0.850855
    "target_2_7",   # 0.857873
    "target_8_2",   # 0.861444
    "target_7_2",   # 0.862925
    "target_1_3",   # 0.875917
    "target_1_5",   # 0.886061
    "target_8_3",   # 0.886388
    "target_1_1",   # 0.910663
    "target_3_2",   # 0.914475
    "target_6_5",   # 0.915495
    "target_9_4",   # 0.917725
    "target_9_8",   # 0.931417
    "target_3_4",   # 0.936777
    "target_2_2",   # 0.939379
    "target_3_5",   # 0.968182
    "target_8_1",   # 0.979974
    "target_2_8"    # 0.997687
]

# Подбор параметров

In [6]:
SAVE_DIR = "optuna_artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)


def feature_importance(model):
    summ_imp = model.feature_importances_.sum()

    if summ_imp == 0:
        feat_imp_df = pd.DataFrame({
            "feature": model.feature_name_,
            "importance": model.feature_importances_
        })
    else:
        feat_imp_df = pd.DataFrame({
            "feature": model.feature_name_,
            "importance": model.feature_importances_ / summ_imp
        })

    return feat_imp_df.sort_values("importance", ascending=False)


def data_selection(target):
    old_model = joblib.load(f"{path_models_old}\\{target}.pkl")
    feature_imp_table = feature_importance(old_model)
    feature_drop = feature_imp_table.loc[
        feature_imp_table["importance"] == 0, "feature"
    ].tolist()

    train_cur = train.drop(columns=feature_drop, errors="ignore").copy()
    test_cur = test.drop(columns=feature_drop, errors="ignore").copy()

    train_x, val_x, train_y_full, val_y_full = train_test_split(
        train_cur,
        train_t,
        test_size=0.2,
        random_state=42
    )

    train_y = train_y_full[target].copy()
    val_y = val_y_full[target].copy()

    return train_x, train_y, val_x, val_y, test_cur, feature_drop


def objective(trial, train_x, train_y, val_x, val_y):
    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",

        "learning_rate": 0.04,
        "num_leaves": trial.suggest_int("num_leaves", 31, 127),
        "max_depth": trial.suggest_int("max_depth", 5, 11),
        "min_child_samples": trial.suggest_int("min_child_samples", 30, 250),
        "subsample": trial.suggest_float("subsample", 0.6, 0.95),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 0.7),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 30.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 1e-4, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 2000, 7000),

        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    }

    pos = train_y.sum()
    neg = len(train_y) - pos
    scale_pos_weight = neg / max(pos, 1)
    scale_pos_weight = min(max(scale_pos_weight, 1.0), 30.0)
    params["scale_pos_weight"] = scale_pos_weight

    model = lgb.LGBMClassifier(**params)

    model.fit(
        train_x,
        train_y,
        eval_set=[(val_x, val_y)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    pred = model.predict_proba(val_x)[:, 1]
    auc = roc_auc_score(val_y, pred)
    return auc


def tune_one_target(target, n_trials=7):
    print(f"\n===== {target} =====")

    train_x, train_y, val_x, val_y, _, feature_drop = data_selection(target)

    study = optuna.create_study(direction="maximize")
    study.optimize(
        lambda trial: objective(trial, train_x, train_y, val_x, val_y),
        n_trials=n_trials,
        show_progress_bar=True,
    )

    best_params = study.best_params.copy()

    pos = train_y.sum()
    neg = len(train_y) - pos
    scale_pos_weight = neg / max(pos, 1)
    scale_pos_weight = min(max(scale_pos_weight, 1.0), 100.0)

    best_params_full = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "learning_rate": 0.04,
        "subsample_freq": 1,
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
        "scale_pos_weight": scale_pos_weight,
        **best_params,
    }

    best_model = lgb.LGBMClassifier(**best_params_full)
    best_model.fit(
        train_x,
        train_y,
        eval_set=[(val_x, val_y)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(100),
        ],
    )

    pred = best_model.predict_proba(val_x)[:, 1]
    best_auc = roc_auc_score(val_y, pred)

    target_dir = os.path.join(SAVE_DIR, target)
    os.makedirs(target_dir, exist_ok=True)

    # study целиком
    joblib.dump(study, os.path.join(target_dir, "study.pkl"))

    # лучшая модель на split
    joblib.dump(best_model, os.path.join(target_dir, "best_model_valid.pkl"))

    # feature_drop
    with open(os.path.join(target_dir, "feature_drop.json"), "w", encoding="utf-8") as f:
        json.dump(feature_drop, f, ensure_ascii=False, indent=2)

    # параметры и служебная инфа
    meta = {
        "target": target,
        "best_auc": float(best_auc),
        "best_params": best_params_full,
        "n_features_after_drop": int(train_x.shape[1]),
        "dropped_features_count": int(len(feature_drop)),
        "dropped_features_file": "feature_drop.json",
    }

    with open(os.path.join(target_dir, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f"best_auc = {best_auc:.6f}")
    print(f"saved to: {target_dir}")

    return {
        "target": target,
        "best_auc": best_auc,
        "best_params": best_params_full,
        "feature_drop": feature_drop,
        "study": study,
        "model": best_model,
    }

In [7]:
for target in targets_sorted[:10]:
    tune_one_target(target)


===== target_9_3 =====


[I 2026-03-29 21:20:14,488] A new study created in memory with name: no-name-4b27e5f0-fccf-47a8-8076-9e56fcb00f09
Best trial: 0. Best value: 0.683946:  14%|█▍        | 1/7 [02:31<15:09, 151.60s/it]

[I 2026-03-29 21:22:46,113] Trial 0 finished with value: 0.6839461188202485 and parameters: {'num_leaves': 87, 'max_depth': 5, 'min_child_samples': 180, 'subsample': 0.8304847578936477, 'colsample_bytree': 0.657428777487671, 'reg_alpha': 0.0035528961402371572, 'reg_lambda': 11.675789000446816, 'min_split_gain': 0.005352349644984665, 'n_estimators': 6313}. Best is trial 0 with value: 0.6839461188202485.


Best trial: 1. Best value: 0.68535:  29%|██▊       | 2/7 [04:25<10:47, 129.45s/it] 

[I 2026-03-29 21:24:40,071] Trial 1 finished with value: 0.6853496038867992 and parameters: {'num_leaves': 50, 'max_depth': 7, 'min_child_samples': 137, 'subsample': 0.9231811855358889, 'colsample_bytree': 0.334377618364203, 'reg_alpha': 0.0014955667626804347, 'reg_lambda': 15.965793175308386, 'min_split_gain': 0.054906611460964504, 'n_estimators': 3617}. Best is trial 1 with value: 0.6853496038867992.


...

[I 2026-03-30 01:40:01,038] Trial 4 finished with value: 0.7306032244999208 and parameters: {'num_leaves': 46, 'max_depth': 10, 'min_child_samples': 153, 'subsample': 0.7823785175194011, 'colsample_bytree': 0.37808868487711433, 'reg_alpha': 0.001003224707301675, 'reg_lambda': 0.01809728308970314, 'min_split_gain': 0.0017447767436061356, 'n_estimators': 6367}. Best is trial 3 with value: 0.7384971752643319.


Best trial: 3. Best value: 0.738497:  86%|████████▌ | 6/7 [16:31<02:28, 148.38s/it]

[I 2026-03-30 01:42:08,681] Trial 5 finished with value: 0.733623925812461 and parameters: {'num_leaves': 41, 'max_depth': 6, 'min_child_samples': 226, 'subsample': 0.7680093200909813, 'colsample_bytree': 0.4444120478905116, 'reg_alpha': 0.5094241267454521, 'reg_lambda': 0.01542860136021977, 'min_split_gain': 0.06164673964637751, 'n_estimators': 6397}. Best is trial 3 with value: 0.7384971752643319.


Best trial: 3. Best value: 0.738497: 100%|██████████| 7/7 [19:14<00:00, 164.99s/it]


[I 2026-03-30 01:44:51,969] Trial 6 finished with value: 0.7363591524260138 and parameters: {'num_leaves': 123, 'max_depth': 5, 'min_child_samples': 147, 'subsample': 0.8103642828204439, 'colsample_bytree': 0.5047686243518832, 'reg_alpha': 8.929250871224898, 'reg_lambda': 14.888502632785173, 'min_split_gain': 0.004416716963230976, 'n_estimators': 6155}. Best is trial 3 with value: 0.7384971752643319.
[100]	valid_0's auc: 0.71678
[200]	valid_0's auc: 0.721823
[300]	valid_0's auc: 0.720876
best_auc = 0.722714
saved to: optuna_artifacts\target_2_4
